In [1]:
!pip install transformers
!pip install datasets
!pip install evaluate
!pip install accelerate -U

import transformers
from transformers import pipeline
from transformers import AutoTokenizer
from transformers import AutoModel
from transformers import AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments
from transformers import Trainer
from transformers import AdamW
from transformers import get_scheduler
import torch
from torch.utils.data import DataLoader
import evaluate
import numpy as np
from datasets import load_dataset
from tqdm.auto import tqdm

**simple usage**

In [2]:
#文章がpositiveかnegativeか分類
checkpoint="distilbert-base-uncased-finetuned-sst-2-english"
tokenizer=AutoTokenizer.from_pretrained(checkpoint)
model=AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequences=["I've been waiting for a HuggingFace course my whole life.", "So have I!"]
tokens=tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

ids=tokenizer.convert_tokens_to_ids(tokens)
input_ids=torch.tensor([ids])
print("Input IDs:", input_ids)

output=model(**tokens)
print(output)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Input IDs: tensor([[100, 100]])
SequenceClassifierOutput(loss=None, logits=tensor([[-1.5607,  1.6123],
        [-3.6183,  3.9137]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


**fine tuning**


In [3]:
#データセットの確認
datasets=load_dataset("glue", "mrpc")
datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [4]:
#trainデータセット
train_dataset=datasets["train"]
train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [5]:
train_dataset.features

{'sentence1': Value(dtype='string', id=None),
 'sentence2': Value(dtype='string', id=None),
 'label': ClassLabel(names=['not_equivalent', 'equivalent'], id=None),
 'idx': Value(dtype='int32', id=None)}

In [6]:
#入力するデータの準備
checkpoint="bert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(checkpoint)

def torknize_func(example):
  return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets=datasets.map(torknize_func, batched=True)
data_collator=DataCollatorWithPadding(tokenizer=tokenizer)

tokenized_datasets["train"].column_names

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

['sentence1',
 'sentence2',
 'label',
 'idx',
 'input_ids',
 'token_type_ids',
 'attention_mask']

In [7]:
#trainingの設定
training_args=TrainingArguments("test-trainer")
model=AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)
trainer=Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
#training
trainer.train()

Step,Training Loss
500,0.561000
1000,0.376900


TrainOutput(global_step=1377, training_loss=0.40168052457256775, metrics={'train_runtime': 202.3624, 'train_samples_per_second': 54.378, 'train_steps_per_second': 6.805, 'total_flos': 405114969714960.0, 'train_loss': 0.40168052457256775, 'epoch': 3.0})

In [9]:
#validationデータセットで予測のテスト
predictions=trainer.predict(tokenized_datasets["validation"])
preds=np.argmax(predictions.predictions, axis=-1)
print("predictions shape:" ,predictions.predictions.shape)
print("pred score:" ,predictions[0][0])
print("pred label:" ,preds[0])

predictions shape: (408, 2)
pred score: [-2.9324918  2.4617906]
pred label: 1


In [10]:
#予測性能の確認
metric=evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

{'accuracy': 0.8357843137254902, 'f1': 0.8870151770657673}

**fine tuning using pytorch**

In [11]:
#入力するデータの準備
raw_datasets=load_dataset("glue", "mrpc")
checkpoint="bert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(checkpoint)

def torknize_func(example):
  return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets=datasets.map(torknize_func, batched=True)
data_collator=DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

In [12]:
tokenized_datasets=tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets=tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [13]:
#dataloaderの作成
train_dataloader=DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader=DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 61]),
 'token_type_ids': torch.Size([8, 61]),
 'attention_mask': torch.Size([8, 61])}

In [14]:
#modelの設定と出力の確認
model=AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
outputs=model(**batch)
print(outputs.loss, outputs.logits.shape)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tensor(0.6597, grad_fn=<NllLossBackward0>) torch.Size([8, 2])


In [15]:
#trainingの設定
optimizer=AdamW(model.parameters(), lr=5e-5)
num_epochs=3
num_training_steps=num_epochs*len(train_dataloader)
lr_scheduler=get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

1377


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [16]:
#deviceの設定
device=torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device

device(type='cuda')

In [17]:
#training
pregress_bar=tqdm(range(num_training_steps))
model.train()

for epoch in range(num_epochs):
  for batch in train_dataloader:
    batch={k: v.to(device) for k, v in batch.items()}
    outputs=model(**batch)
    loss=outputs.loss
    loss.backward()

    optimizer.step()
    lr_scheduler.step()
    optimizer.zero_grad()
    pregress_bar.update(1)

  0%|          | 0/1377 [00:00<?, ?it/s]

In [18]:
#予測性能の確認
metric=evaluate.load("glue", "mrpc")
model.eval()

for batch in eval_dataloader:
  batch={k: v.to(device) for k, v in batch.items()}
  with torch.no_grad():
    outputs=model(**batch)
  logits=outputs.logits
  predictions=torch.argmax(logits,dim=-1)
  metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.8504901960784313, 'f1': 0.8957264957264958}